# TOPSIS
---
_**T**echnique for **O**rder **P**reference by **S**imilarity to **I**deal **S**olution_

In [1]:
import numpy as np
import pandas as pd

In [2]:
import warnings
warnings.filterwarnings("ignore")

## 1. Introduction
---
The **TOPSIS** (_Technique for Order Preference by Similarity to Ideal Solution_) method is a multi-criteria decision analysis technique developed by Hwang and Yoon in 1981. The core idea is that the best alternative should have the **shortest distance** from the _positive ideal solution_ (formed by the best values across all criteria) and the **farthest distance** from the _negative ideal solution_ (formed by the worst values). The overall performance of each alternative is measured through its **relative closeness** to the ideal, which is used to produce a final ranking.

### 1.1. Applications
---
The _TOPSIS method_ can be used in a wide range of decision-making situations, such as:

- Supplier and vendor selection
- Engineering material selection
- Healthcare facility performance evaluation
- Environmental impact assessment

For example, let's consider a set of elements with **4 criteria** and **5 alternatives**, generated randomly:

In [3]:
np.random.seed(0)

M = 5
C = np.random.randint(100, 500, M)
df = pd.DataFrame({
    "Criteria 1": C,
    "Criteria 2": (C**(np.random.random(M)/2)).astype(int),
    "Criteria 3": np.log(C).astype(int),
    "Criteria 4": (C*(1 + np.random.random(M))//2).astype(int)
})

df.index = [f'Alternative {i}' for i in np.arange(M)+1]

df.style.background_gradient()

,Criteria 1,Criteria 2,Criteria 3,Criteria 4
Alternative 1,272,11,5,143
Alternative 2,147,8,4,93
Alternative 3,217,5,5,160
Alternative 4,292,2,5,264
Alternative 5,423,2,6,313


## 2. Method steps
---

### 2.1. Objectives and weights
---
The first step is the definition of **objectives** and **weights**. The _objective_ refers to **minimizing** or **maximizing** the criteria. The _weight_ defines the degree of importance of each criterion. All of them must be defined by the **decision maker**, before proceeding with the method.

In [4]:
# Criteria 1: {objective: max, weight: 4},
# Criteria 2: {objective: min, weight: 2},
# Criteria 3: {objective: max, weight: 3},
# Criteria 4: {objective: max, weight: 3}

OBJECTIVES = pd.Series(["max", "min", "max", "max"])
WEIGHTS = pd.Series([4, 2, 3, 3])
WEIGHTS = WEIGHTS/WEIGHTS.sum()

OBJECTIVES.index = WEIGHTS.index = [
    "Criteria 1", "Criteria 2", "Criteria 3", "Criteria 4"
]

OBJECTIVES, WEIGHTS

(Criteria 1    max
 Criteria 2    min
 Criteria 3    max
 Criteria 4    max
 dtype: object,
 Criteria 1    0.333333
 Criteria 2    0.166667
 Criteria 3    0.250000
 Criteria 4    0.250000
 dtype: float64)

### 2.2. Normalized decision matrix
---
The second step is to normalize the decision matrix using the **vector normalization** method. This approach preserves the relative scale of the data and ensures that all criteria are dimensionless. For each criterion $j$, the normalized value is computed as:

$$ \large
r_{ij} = \frac{a_{ij}}{\sqrt{\sum_{i=1}^m a_{ij}^2}}
$$

where $a_{ij}$ is the value of alternative $i$ for criterion $j$, and $m$ is the number of alternatives.

In [5]:
df_n = df / (df**2).sum(axis=0)**0.5

df_n

,Criteria 1,Criteria 2,Criteria 3,Criteria 4
Alternative 1,0.426421,0.745014,0.443678,0.303253
Alternative 2,0.230455,0.541828,0.354943,0.197220
Alternative 3,0.340196,0.338643,0.443678,0.339304
Alternative 4,0.457775,0.135457,0.443678,0.559851
Alternative 5,0.663147,0.135457,0.532414,0.663763


### 2.3. Weighted normalized decision matrix
---
The third step is to multiply each normalized value by the corresponding criterion weight, so that criteria with higher importance have a greater influence on the final result:

$$ \large
v_{ij} = w_j \cdot r_{ij}
$$

where $w_j$ is the weight of criterion $j$, such that $\sum_j w_j = 1$.

In [6]:
df_w = df_n * WEIGHTS

df_w

,Criteria 1,Criteria 2,Criteria 3,Criteria 4
Alternative 1,0.142140,0.124169,0.110920,0.075813
Alternative 2,0.076818,0.090305,0.088736,0.049305
Alternative 3,0.113399,0.056440,0.110920,0.084826
Alternative 4,0.152592,0.022576,0.110920,0.139963
Alternative 5,0.221049,0.022576,0.133103,0.165941


### 2.4. Ideal best and ideal worst solutions
---
The fourth step is to determine the **positive ideal solution** $A^+$ (ideal best) and the **negative ideal solution** $A^-$ (ideal worst). The positive ideal solution is formed by selecting the best value for each criterion across all alternatives, and the negative ideal solution by selecting the worst.

For criteria the objective is **maximization**:

$$ \large
A_j^+ = \max_i v_{ij} \qquad A_j^- = \min_i v_{ij}
$$

For criteria the objective is **minimization**:

$$ \large
A_j^+ = \min_i v_{ij} \qquad A_j^- = \max_i v_{ij}
$$

In [7]:
A_pos = {}
A_neg = {}

for col in df_w.columns:
    if OBJECTIVES[col] == "max":
        A_pos[col] = df_w[col].max()
        A_neg[col] = df_w[col].min()
    else:
        A_pos[col] = df_w[col].min()
        A_neg[col] = df_w[col].max()

A_pos = pd.Series(A_pos, name="Ideal best")
A_neg = pd.Series(A_neg, name="Ideal worst")

pd.DataFrame([A_pos, A_neg])

,Criteria 1,Criteria 2,Criteria 3,Criteria 4
Ideal best,0.221049,0.022576,0.133103,0.165941
Ideal worst,0.076818,0.124169,0.088736,0.049305


### 2.5. Euclidean distances
---
The fifth step is to calculate the **Euclidean distance** of each alternative from the positive ideal solution $d_i^+$ and from the negative ideal solution $d_i^-$:

$$ \large
d_i^+ = \sqrt{\sum_{j}^n \left(v_{ij} - A_j^+\right)^2}
$$

$$ \large
d_i^- = \sqrt{\sum_{j}^n \left(v_{ij} - A_j^-\right)^2}
$$

A small $d_i^+$ means the alternative is close to the ideal best, while a large $d_i^-$ means it is far from the ideal worst — both are desirable.

In [8]:
d_pos = ((df_w - A_pos)**2).sum(axis=1)**0.5
d_neg = ((df_w - A_neg)**2).sum(axis=1)**0.5

pd.DataFrame({"d+": d_pos, "d-": d_neg})

,d+,d-
Alternative 1,0.158628,0.073904
Alternative 2,0.202391,0.033864
Alternative 3,0.140738,0.087631
Alternative 4,0.076507,0.157396
Alternative 5,0.000000,0.216092


### 2.6. Relative closeness
---
The sixth step is to calculate the **relative closeness** $C_i$ of each alternative to the positive ideal solution. This value is bounded between 0 and 1, where a value closer to 1 indicates a better alternative:

$$ \large
C_i = \frac{d_i^-}{d_i^+ + d_i^-}
$$

When $C_i = 1$, the alternative coincides with the positive ideal solution. When $C_i = 0$, it coincides with the negative ideal solution.

In [9]:
score = d_neg / (d_pos + d_neg)
score = score.to_frame("score")

score

,score
Alternative 1,0.317822
Alternative 2,0.143338
Alternative 3,0.383725
Alternative 4,0.672911
Alternative 5,1.000000


### 2.7. Ranking
---
Finally, the last step is to define a **ranking** by ordering the _relative closeness_ in descending way.

In [10]:
df_score = df.join(score)
df_score["ranking"] = (
    df_score["score"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)

(
    df_score.sort_values("ranking")
    .style.background_gradient(
        subset=list(df.columns)
    )
)

,Criteria 1,Criteria 2,Criteria 3,Criteria 4,score,ranking
Alternative 5,423,2,6,313,1.000000,1
Alternative 4,292,2,5,264,0.672911,2
Alternative 3,217,5,5,160,0.383725,3
Alternative 1,272,11,5,143,0.317822,4
Alternative 2,147,8,4,93,0.143338,5


## 3. Examples
---

### 3.1. Laptop selection
---
As an example, let's select the best laptop by comparing them based on their specifications. The data was taken from [notebookcheck](https://www.notebookcheck.net/) and we will consider 5 criteria: **price** (in USD), **RAM** (in GB), **storage** (in GB), **battery life** (in hours) and **weight** (in kg). The _price_ and _weight_ are criteria we want to **minimize**.

In [11]:
df_ex1 = pd.DataFrame({
    "price": [1999, 1299, 1649, 1599, 1499, 1099],
    "RAM": [16, 32, 16, 16, 8, 16],
    "storage": [512, 512, 512, 1024, 256, 512],
    "battery": [17, 13, 15, 17, 18, 11],
    "weight": [1.55, 1.86, 1.12, 1.36, 1.29, 1.39]
})

df_ex1.index = [
    "Apple MacBook Pro 14",
    "Dell XPS 15",
    "Lenovo ThinkPad X1 Carbon",
    "HP Spectre x360 14",
    "Microsoft Surface Laptop 5",
    "ASUS ZenBook 14"
]

df_ex1.style.background_gradient()

,price,RAM,storage,battery,weight
Apple MacBook Pro 14,1999,16,512,17,1.550000
Dell XPS 15,1299,32,512,13,1.860000
Lenovo ThinkPad X1 Carbon,1649,16,512,15,1.120000
HP Spectre x360 14,1599,16,1024,17,1.360000
Microsoft Surface Laptop 5,1499,8,256,18,1.290000
ASUS ZenBook 14,1099,16,512,11,1.390000


In [12]:
# Objective and weights
#    price: {objective: min, weight: 5},
#      RAM: {objective: max, weight: 3},
#  storage: {objective: max, weight: 2},
#  battery: {objective: max, weight: 4},
#   weight: {objective: min, weight: 3}

OBJECTIVES_ex1 = pd.Series(["min", "max", "max", "max", "min"])
WEIGHTS_ex1 = pd.Series([5, 3, 2, 4, 3])
WEIGHTS_ex1 = WEIGHTS_ex1/WEIGHTS_ex1.sum()

OBJECTIVES_ex1.index = WEIGHTS_ex1.index = [
    "price", "RAM", "storage", "battery", "weight"
]

# Normalized decision matrix
df_ex1_n = df_ex1 / (df_ex1**2).sum(axis=0)**0.5

# Weighted normalized decision matrix
df_ex1_w = df_ex1_n * WEIGHTS_ex1

# Ideal best and ideal worst solutions
A_pos_ex1 = {}
A_neg_ex1 = {}

for col in df_ex1_w.columns:
    if OBJECTIVES_ex1[col] == "max":
        A_pos_ex1[col] = df_ex1_w[col].max()
        A_neg_ex1[col] = df_ex1_w[col].min()
    else:
        A_pos_ex1[col] = df_ex1_w[col].min()
        A_neg_ex1[col] = df_ex1_w[col].max()

A_pos_ex1 = pd.Series(A_pos_ex1)
A_neg_ex1 = pd.Series(A_neg_ex1)

# Euclidean distances
d_pos_ex1 = ((df_ex1_w - A_pos_ex1)**2).sum(axis=1)**0.5
d_neg_ex1 = ((df_ex1_w - A_neg_ex1)**2).sum(axis=1)**0.5

# Relative closeness
score_ex1 = d_neg_ex1 / (d_pos_ex1 + d_neg_ex1)

# Ranking
df_ex1["score"] = score_ex1
df_ex1["ranking"] = (
    df_ex1["score"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)

(
    df_ex1.sort_values("ranking")
    .style.background_gradient(
        subset=["price", "RAM", "storage", "battery", "weight"]
    )
)

,price,RAM,storage,battery,weight,score,ranking
Dell XPS 15,1299,32,512,13,1.860000,0.626951,1
HP Spectre x360 14,1599,16,1024,17,1.360000,0.543015,2
ASUS ZenBook 14,1099,16,512,11,1.390000,0.486446,3
Lenovo ThinkPad X1 Carbon,1649,16,512,15,1.120000,0.422579,4
Microsoft Surface Laptop 5,1499,8,256,18,1.290000,0.360330,5
Apple MacBook Pro 14,1999,16,512,17,1.550000,0.345323,6
